# [Programming Z3](https://theory.stanford.edu/~nikolaj/programmingz3.html?utm_source=chatgpt.com#sec-logical-interfaces)

## 2. Logical Interfaces to Z3

In [5]:
from z3 import Bool, BoolRef, Not, Or, Solver, sat

tie: BoolRef = Bool("Tie")
shirt: BoolRef = Bool("Shirt")

s: Solver = Solver()
s.add(
    Or(tie, shirt),
    Or(Not(tie), shirt),
    Or(Not(tie), Not(shirt)),
)

result = s.check()
print(result)

if result == sat:
    print(s.model())

sat
[Tie = False, Shirt = True]


In [ ]:
from z3 import (
    Array,
    ArrayRef,
    ArithRef,
    BoolRef,
    FuncDeclRef,
    Function,
    Implies,
    Int,
    IntSort,
    Not,
    Solver,
    SortRef,
    Store,
)

Z: SortRef = IntSort()

f: FuncDeclRef = Function("f", Z, Z)  # 非解釈関数

x: ArithRef = Int("x")
y: ArithRef = Int("y")
z: ArithRef = Int("z")

A: ArrayRef = Array("A", Z, Z)

fml: BoolRef = Implies(
    x + 2 == y,
    # Store(A, x, 3) は、配列 A の添字 x に値 3 を格納した新しい配列を表す
    f(Store(A, x, 3)[y - 2]) == f(y - x + 1),
)

s: Solver = Solver()
s.add(Not(fml))  # 否定

print(s.check())  # unsat

unsat


In [7]:
from z3 import ArithRef, Int, Solver

x: ArithRef = Int("x")
y: ArithRef = Int("y")

s: Solver = Solver()

s.add((x % 4) + 3 * (y / 2) > x - y)

print(s.sexpr())

(declare-fun y () Int)
(declare-fun x () Int)
(assert (> (+ (mod x 4) (* 3 (div y 2))) (- x y)))



In [8]:
from z3 import Bool, BoolRef, Implies, Not, Solver

a: BoolRef = Bool("a")
b: BoolRef = Bool("b")
r: BoolRef = Bool("r")

s: Solver = Solver()
s.add(
    Implies(a, r),
    Implies(b, r),
    a,
)

# 現在の公理集合から r が導けるか？
s.push()
s.add(Not(r))

print(s.check())  # unsat

s.pop()

unsat


### 2.1. Sorts

$$
S^{\mathcal M}\neq\varnothing
$$

となる議論領域を持つ Sort \(S\) を宣言したことに

In [ ]:
from z3 import (
    BoolRef,
    Const,
    DeclareSort,
    ExprRef,
    ForAll,
    Solver,
    SortRef,
)

S: SortRef = DeclareSort("S")  # 新しい Sort を定義
s: ExprRef = Const("s", S)  # Sort S に属する定数記号 s の宣言

solver: Solver = Solver()
solver.add(ForAll([s], s != s))

print(solver.check())  # unsat

unsat


### 2.2 Signatures


$$
f:\mathbb Z\to\mathbb Z
$$

$$
P:\mathbb Z\to\mathbb B
$$

Z3 では Predicate も Boolean-valued Function として宣言

In [13]:
from z3 import (
    BoolSort,
    ExprRef,
    FuncDeclRef,
    Function,
    IntSort,
    SortRef,
)

Z: SortRef = IntSort()

f: FuncDeclRef = Function("f", Z, Z)  # 関数記号
P: FuncDeclRef = Function("P", Z, BoolSort())  # 述語記号

term = f(x)  # 項

### 2.3 Terms and Formulas

In [ ]:
from z3 import (
    Bool,
    BoolRef,
    BoolSort,
    FuncDeclRef,
    Function,
    IntSort,
    Solver,
    SortRef,
)

B: SortRef = BoolSort()
Z: SortRef = IntSort()

f: FuncDeclRef = Function("f", B, Z)
g: FuncDeclRef = Function("g", Z, B)

a: BoolRef = Bool("a")  # 定数記号

solver: Solver = Solver()
solver.add(g(1 + f(a)))

print(solver.check())  # sat
print(solver.model())

sat
[a = False, g = [else -> True], f = [else -> 0]]


In [15]:
from z3 import ArithRef, BoolRef, Int

x: ArithRef = Int("x")
y: ArithRef = Int("y")

n: BoolRef = x + y >= 3

print("num args:", n.num_args())
print("children:", n.children())
print("1st child:", n.arg(0))
print("2nd child:", n.arg(1))
print("operator:", n.decl())
print("op name:", n.decl().name())

num args: 2
children: [x + y, 3]
1st child: x + y
2nd child: 3
operator: >=
op name: >=


### 2.4. Quantifiers and Lambda binding

$$
\underbrace{y=x+1}_{\text{外側の制約}}
\quad\land\quad
\underbrace{\forall y\;(y\le 0\to x<y)}_{\text{量化式}}
$$

ここで、外側の $y$ と量化式の内側の $y$ は、別の役割

In [16]:
from z3 import ArithRef, BoolRef, ForAll, Implies, Int, Solver

x: ArithRef = Int("x")
y: ArithRef = Int("y")

# 例1
phi1: BoolRef = ForAll(
    [y],
    Implies(y <= 0, x < y),
)

s1: Solver = Solver()
s1.add(y == x + 1, phi1)
print(s1.check())  # unsat


unsat


In [17]:
# 例2
phi2: BoolRef = ForAll(
    [y],
    Implies(y <= 0, x > y),
)

s2: Solver = Solver()
s2.add(y == x + 1, phi2)
print(s2.check())  # sat
print(s2.model())

sat
[x = 2, y = 3]


#### lambda

$$
m'=
\lambda x.
\begin{cases}
v & l\le x\le h\\
m(x) & \text{otherwise}
\end{cases}
$$

ここで
- $x$：Lambda が束縛する変数
- $l,h$：更新範囲
- $v$：設定したい値
- $m$：元の配列
- $m'$：新しく構築した配列

In [ ]:
from z3 import (
    And,
    ArithRef,
    Array,
    ArrayRef,
    If,
    Int,
    IntSort,
    Lambda,
    QuantifierRef,
    Select,
    Solver,
)

m: ArrayRef = Array("m", IntSort(), IntSort())
m1: ArrayRef = Array("m1", IntSort(), IntSort())
z: ArithRef = Int("z")


def memset(
    lo: int,
    hi: int,
    value: ArithRef,
    base: ArrayRef,
) -> QuantifierRef:
    x: ArithRef = Int("x")
    
    # Lambda：変数を束縛して関数を構築する
    return Lambda(
        [x],
        If(
            And(lo <= x, x <= hi),
            value,
            Select(base, x),
        ),
    )


updated: QuantifierRef = memset(1, 700, z, m)

s: Solver = Solver()
s.add(
    m1 == updated,
    Select(m1, 6) != z,
)
print(s.check())  # unsat

unsat


#### 述語を量化

In [ ]:
from z3 import (
    ArithRef,
    Array,
    ArrayRef,
    BoolRef,
    BoolSort,
    ForAll,
    Implies,
    Int,
    IntSort,
    Not,
    Select,
    Solver,
)

x: ArithRef = Int("x")
y: ArithRef = Int("y")

# 一部の二階論理的な推論を表現できる
Q: ArrayRef = Array("Q", IntSort(), BoolSort())

phi: BoolRef = Implies(
    ForAll(
        [Q],
        Implies(Select(Q, x), Select(Q, y)),
    ),
    x == y,
)

s: Solver = Solver()
s.add(Not(phi))

print(s.check())  # unsat
